# FACS-sorted ONT fractions (HCT116) vs APEX-seq (HEK293T): fracSanity patterns compared

Gene by gene comparison of the fracSanity results of `ONT_organelles.fracSanity.ipynb` (FACS-sorted subcellular fractions of HCT116, ONT, siCtrl, reference: Total RNA) and `APEX_seq.fracSanity.ipynb` (APEX-seq of Fazal et al. 2019, HEK293T, Illumina, reference: 18 unlabelled controls). Both use the same model and the same quantity: $r_g = \log_2\alpha_g - \text{median}$ of the cohort, the over- (> 0) or under-representation (< 0) of the mRNA of gene $g$ in the fraction relative to the average gene, which does not depend on the unidentified global factor $F$.

**Matching fractions.**

| ONT fraction (FACS sort) | APEX-seq construct | expectation |
|:---|:---|:---|
| ER (CANX), TIS (ZFP36L1: TIS granules on the ER) | ERM (ER membrane, cytosolic face), KDEL (ER lumen) | mRNAs translated at the ER (secretory and membrane proteins) enriched in both |
| Free (no organelle marker) | NES (cytosol), cytosolic fraction | ER-translated mRNAs depleted, cytosolic mRNAs (e.g. ribosomal proteins) enriched |
| Golgi, PEX, Endo, TJ (membrane sorts) | ERM, OMM | membrane-associated mRNAs; no APEX construct for Golgi, peroxisomes, endosomes or tight junctions |
| nuclear contamination of the membrane sorts (MALAT1) | NLS, nuclear fraction | nuclear RNAs and pre-mRNAs enriched |

**Caveats.** Different cell lines (HCT116 vs HEK293T: different expressed genes), different principles (sorting of particles after lysis vs labelling in living cells within ~20 nm of the construct), different references (Total RNA vs bead background), different reads (full-length ONT molecules vs short reads; PAQR coverage vs read 3' ends for APA). Correlations are therefore expected to be modest; what matters is which pairs correlate most and whether the matching pairs stand out.

Inputs: the per-cohort tables of the two notebooks (`alpha_genes/fracSanity_alpha.genes.per_cohort.tsv.gz`, `gene_sets/gene_set_activities.per_cohort.tsv.gz`, `apa/utr_length.fraction_vs_Total.tsv.gz`). Genes: Ensembl IDs without version, both GENCODE v42 basic; genes with $\alpha_g$ in both data sets and ≥ 10 CPM (ONT) / ≥ 10 TPM (APEX-seq) in the references. The two expression scales correspond: ONT reads are unfragmented full-length molecules (CPM = share of molecules), and the APEX-seq Sanity runs are length-corrected (TPM).

In [ ]:
import warnings
warnings.simplefilter('ignore')

import os
import csv

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import spearmanr

from dotenv import load_dotenv

import seaborn as sns; sns.set()
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML
%matplotlib inline

In [ ]:
load_dotenv("APA_localization.scicore.env", override=True)
subdirs = {"figures_dir": os.getenv("FIGURES_DIR"), "tables_dir": os.getenv("TABLES_DIR")}
ANALYSIS = "APEX_vs_ONT_organelles_fracSanity"
T_ONT = os.path.join(subdirs["tables_dir"], "gene_expression_analysis", "ONT_organelles_fracSanity")
T_APEX = os.path.join(subdirs["tables_dir"], "gene_expression_analysis", "APEX_seq_fracSanity")
FIGS = os.path.join(subdirs["figures_dir"], "gene_expression_analysis", ANALYSIS)
TABLES = os.path.join(subdirs["tables_dir"], "gene_expression_analysis", ANALYSIS)
for d in (FIGS, TABLES):
    os.makedirs(d, exist_ok=True)

def figure_caption(path, formats=("png", "pdf")):
    path = os.path.splitext(path)[0] if path.endswith((".png", ".pdf")) else path
    rel = os.path.relpath(path, subdirs["figures_dir"])
    links = " | ".join(f'<a href="figures/{rel}.{f}" target="_blank">{f}</a>' for f in formats)
    display(HTML(f'<p><b>Figure:</b> <code>{os.path.basename(path)}</code> &nbsp;({links})</p>'))

def savefig(fig, name, formats=("png", "pdf")):
    for fmt in formats:
        fig.savefig(os.path.join(FIGS, f"{name}.{fmt}"), bbox_inches="tight", dpi=300)
    figure_caption(os.path.join(FIGS, name), formats)

def save_tsv(df, name, index=False):
    df.to_csv(os.path.join(TABLES, name), sep="\t", header=True, index=index, quoting=csv.QUOTE_NONE)

FRACTION_COLORS = {"Free": "tab:blue", "Org2": "tab:orange", "ER": "tab:green", "TIS": "tab:red", "Golgi": "tab:purple", "PEX": "tab:brown",
                   "Endo": "tab:pink", "TJ": "tab:olive",
                   "NLS": "tab:red", "NIK": "darkred", "LMA": "tab:orange", "NucPore": "goldenrod", "NES": "tab:blue", "ERM": "tab:green",
                   "KDEL": "yellowgreen", "OMM": "tab:purple", "MITO": "tab:pink", "Nuc": "tab:brown", "Cyto": "tab:cyan"}
ONT_ORDER = ["ER/TIS: Free", "ER/TIS: Org2", "ER/TIS: ER", "ER/TIS: TIS", "Golgi/PEX: Free", "Golgi/PEX: Org2", "Golgi/PEX: Golgi", "Golgi/PEX: PEX",
             "End/TJ: Free", "End/TJ: Org2", "End/TJ: Endo", "End/TJ: TJ"]
APEX_ORDER = ["APEX: NLS", "APEX: NIK", "APEX: LMA", "APEX: NucPore", "Fractionation: Nuc", "APEX: NES", "Fractionation: Cyto", "APEX: ERM",
              "APEX: KDEL", "APEX: OMM", "APEX: MITO"]
frac_of = lambda c: c.split(": ")[-1]

def color_ticklabels(labels):
    for t in labels:
        t.set_color(FRACTION_COLORS.get(frac_of(t.get_text()), "black"))

## Genes and $r_g$ in both data sets

In [ ]:
def load_alpha(folder):
    a = pd.read_csv(os.path.join(folder, "alpha_genes", "fracSanity_alpha.genes.per_cohort.tsv.gz"), sep="\t")
    a = a.loc[~a["gene_id"].str.endswith("_PAR_Y")]
    a["gene"] = a["gene_id"].str.replace(r"\.\d+$", "", regex=True)
    return a

a_ont, a_apex = load_alpha(T_ONT), load_alpha(T_APEX)
ONT_COH = [c for c in ONT_ORDER if c in set(a_ont["cohort"])]
APEX_COH = [c for c in APEX_ORDER if c in set(a_apex["cohort"])]
r_ont = a_ont.pivot(index="gene", columns="cohort", values="r_rel")[ONT_COH]
r_apex = a_apex.pivot(index="gene", columns="cohort", values="r_rel")[APEX_COH]
call_ont = a_ont.pivot(index="gene", columns="cohort", values="call")[ONT_COH]
call_apex = a_apex.pivot(index="gene", columns="cohort", values="call")[APEX_COH]
cpm_ont = a_ont.groupby("gene")["Total_CPM"].first()
cpm_apex = a_apex.groupby("gene")["Total_TPM"].first()     # length-corrected: TPM of APEX-seq ~ CPM of ONT
gtype = a_ont.groupby("gene")["gene_type"].first().combine_first(a_apex.groupby("gene")["gene_type"].first())
gname = a_ont.groupby("gene")["gene_name"].first().combine_first(a_apex.groupby("gene")["gene_name"].first())
common = r_ont.dropna(how="all").index.intersection(r_apex.dropna(how="all").index)
common = common[(cpm_ont.reindex(common) >= 10).to_numpy() & (cpm_apex.reindex(common) >= 10).to_numpy()]
print(f"ONT: {r_ont.shape[0]:,} genes in {len(ONT_COH)} cohorts; APEX-seq: {r_apex.shape[0]:,} genes in {len(APEX_COH)} cohorts; "
      f"{len(common):,} genes with alpha in both and >= 10 CPM (ONT) / TPM (APEX-seq) in the references")
display(gtype.reindex(common).value_counts().head(8).rename("genes").to_frame())

## Correlation of $r_g$ between every ONT fraction and every APEX-seq construct

Spearman correlation of $r_g$ over the common genes, for all genes and for protein-coding genes only (non-coding RNAs, mostly nuclear, can drive a correlation through the nuclear axis alone). Second panel: the same after removing the nuclear axis, i.e. partial correlations given the fractionation $r_g$ of the nuclear fraction (`Fractionation: Nuc`; residuals of both from a rank regression on it).

In [ ]:
def spearman_matrix(A, B):
    return pd.DataFrame({b: [spearmanr(A[a], B[b], nan_policy="omit")[0] for a in A.columns] for b in B.columns}, index=A.columns)

def residual_ranks(df, z):
    # ranks of each column after removing the linear dependence on the ranks of z (partial Spearman)
    zr = z.rank()
    out = {}
    for c in df.columns:
        y = df[c].rank()
        ok = y.notna() & zr.notna()
        b = np.polyfit(zr[ok], y[ok], 1)
        out[c] = y - (b[0] * zr + b[1])
    return pd.DataFrame(out)

pc = common[(gtype.reindex(common) == "protein_coding").to_numpy()]
nuc_axis = r_apex["Fractionation: Nuc"] if "Fractionation: Nuc" in r_apex else r_apex["APEX: NLS"]
mats = {"all genes": spearman_matrix(r_ont.loc[common], r_apex.loc[common]),
        "protein-coding genes": spearman_matrix(r_ont.loc[pc], r_apex.loc[pc]),
        "protein-coding, nuclear axis removed": spearman_matrix(residual_ranks(r_ont.loc[pc], nuc_axis.loc[pc]),
                                                                residual_ranks(r_apex.loc[pc].drop(columns=["Fractionation: Nuc"], errors="ignore"),
                                                                               nuc_axis.loc[pc]))}
for k, m in mats.items():
    save_tsv(m.rename_axis("ONT cohort").reset_index(), f"spearman_r_rel.ONT_vs_APEX.{k.replace(' ', '_').replace(',', '')}.tsv")

sns.set(font_scale=0.8, style="white")
fig, axes = plt.subplots(1, 3, figsize=(20, 5.2))
lim = max(np.nanmax(np.abs(m.values)) for m in mats.values())
for ax, (k, m) in zip(axes, mats.items()):
    sns.heatmap(m, cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, annot=True, fmt=".2f", annot_kws={"fontsize": 6}, ax=ax,
                cbar=ax is axes[-1], cbar_kws={"label": "Spearman of $r_g$"})
    ax.set(title=f"{k} (n = {len(common) if k == 'all genes' else len(pc):,})", xlabel="APEX-seq", ylabel="ONT (FACS)" if ax is axes[0] else "")
    color_ticklabels(ax.get_xticklabels()); color_ticklabels(ax.get_yticklabels())
fig.tight_layout()
savefig(fig, "spearman_r_rel.ONT_vs_APEX")
plt.show()

## Matching fractions

Scatter of $r_g$ for the matching pairs (protein-coding genes; colour: called in ONT, over / under; marker size: called in APEX-seq too), and the overlap of the called genes: odds ratio (Fisher's exact test) of being over- (or under-) represented in the APEX-seq construct for genes over- (or under-) represented in the ONT fraction, among the common genes called or not in both.

In [ ]:
PAIRS = [("ER/TIS: ER", "APEX: ERM"), ("ER/TIS: ER", "APEX: KDEL"), ("ER/TIS: TIS", "APEX: ERM"), ("ER/TIS: Free", "APEX: NES"),
         ("Golgi/PEX: Free", "APEX: NES"), ("End/TJ: Free", "APEX: NES"), ("Golgi/PEX: Golgi", "APEX: ERM"), ("End/TJ: Endo", "APEX: ERM"),
         ("End/TJ: TJ", "APEX: NLS"), ("Golgi/PEX: PEX", "APEX: OMM")]
PAIRS = [(o, a) for o, a in PAIRS if o in r_ont and a in r_apex]

ov_rows = []
for o, a in PAIRS:
    co, ca = call_ont.loc[common, o], call_apex.loc[common, a]
    for d in ("over", "under"):
        t = pd.crosstab(co == d, ca == d).reindex(index=[True, False], columns=[True, False], fill_value=0)
        orr, p = stats.fisher_exact(t.values)
        ov_rows.append({"ONT": o, "APEX": a, "direction": d, "both": int(t.loc[True, True]), "ONT_only": int(t.loc[True, False]),
                        "APEX_only": int(t.loc[False, True]), "odds_ratio": orr, "p_value": p,
                        "spearman_r_rel_pc": spearmanr(r_ont.loc[pc, o], r_apex.loc[pc, a], nan_policy="omit")[0]})
ov = pd.DataFrame(ov_rows)
save_tsv(ov, "matching_pairs.called_gene_overlap.tsv")
display(ov.round(4))

ncol = 5
nrow = int(np.ceil(len(PAIRS) / ncol))
sns.set(font_scale=0.8, style="ticks")
fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 3.2 * nrow), squeeze=False)
CALL_COLORS = {"over": "tab:red", "under": "tab:blue", "n.s.": "lightgrey"}
for ax, (o, a) in zip(axes.flat, PAIRS):
    x, y = r_ont.loc[pc, o], r_apex.loc[pc, a]
    c = call_ont.loc[pc, o].fillna("n.s.")
    both = (call_apex.loc[pc, a] == c) & (c != "n.s.")
    for k in ("n.s.", "under", "over"):
        m = (c == k).to_numpy()
        ax.scatter(x[m], y[m], s=np.where(both[m], 9, 2), color=CALL_COLORS[k], lw=0, alpha=0.5 if k == "n.s." else 0.8, rasterized=True)
    ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
    ax.set_title(f"Spearman {spearmanr(x, y, nan_policy='omit')[0]:.2f}", fontsize=8)
    ax.set_xlabel(f"$r_g$ ONT {o}", color=FRACTION_COLORS[frac_of(o)])
    ax.set_ylabel(f"$r_g$ {a}", color=FRACTION_COLORS[frac_of(a)])
for ax in axes.flat[len(PAIRS):]:
    ax.axis("off")
fig.legend(handles=[plt.Line2D([], [], color=v, marker="o", ls="", label=f"ONT {k}") for k, v in CALL_COLORS.items()],
           loc="upper right", fontsize=7, frameon=False)
sns.despine(); fig.tight_layout()
savefig(fig, "matching_pairs.r_rel_scatter")
plt.show()

## Gene-set activities

Activities of the same sets (same database fits, with the same covariates: biotypes and expression deciles of the reference) in the ONT fractions and the APEX-seq constructs: Spearman correlation of the activities over all sets of a database, per pair of cohorts, and the activities of the sets that are significant (padj < 0.05) in at least two cohorts of each data set, side by side.

In [ ]:
def load_act(folder):
    return pd.read_csv(os.path.join(folder, "gene_sets", "gene_set_activities.per_cohort.tsv.gz"), sep="\t")

act_o, act_a = load_act(T_ONT), load_act(T_APEX)
DBS = [d for d in ["GO:CC", "GO:BP", "Reactome", "KEGG"] if d in set(act_o["fit"]) and d in set(act_a["fit"])]
rows = []
for db in DBS:
    mo = act_o.loc[(act_o["fit"] == db) & ~act_o["set_id"].str.startswith(("BIOTYPE:", "EXPR:"))].pivot(index="set_id", columns="cohort", values="activity")
    ma = act_a.loc[(act_a["fit"] == db) & ~act_a["set_id"].str.startswith(("BIOTYPE:", "EXPR:"))].pivot(index="set_id", columns="cohort", values="activity")
    s = mo.index.intersection(ma.index)
    m = spearman_matrix(mo.loc[s, [c for c in ONT_COH if c in mo]], ma.loc[s, [c for c in APEX_COH if c in ma]])
    rows.append(m.stack().rename("spearman").reset_index().rename(columns={"level_0": "ONT", "cohort": "APEX"}).assign(database=db, n_sets=len(s)))
act_corr = pd.concat(rows, ignore_index=True)
act_corr.columns = ["ONT", "APEX", "spearman", "database", "n_sets"]
save_tsv(act_corr, "gene_set_activities.spearman.ONT_vs_APEX.tsv")

sns.set(font_scale=0.75, style="white")
fig, axes = plt.subplots(1, len(DBS), figsize=(6.2 * len(DBS), 5), squeeze=False)
for ax, db in zip(axes[0], DBS):
    m = act_corr.loc[act_corr["database"] == db].pivot(index="ONT", columns="APEX", values="spearman").reindex(index=ONT_COH, columns=APEX_COH)
    sns.heatmap(m, cmap="RdBu_r", center=0, vmin=-1, vmax=1, annot=True, fmt=".2f", annot_kws={"fontsize": 5.5}, ax=ax, cbar=ax is axes[0, -1])
    ax.set(title=f"{db}: {int(act_corr.loc[act_corr['database'] == db, 'n_sets'].iloc[0]):,} sets", xlabel="APEX-seq", ylabel="")
    color_ticklabels(ax.get_xticklabels()); color_ticklabels(ax.get_yticklabels())
fig.tight_layout()
savefig(fig, "gene_set_activities.spearman.ONT_vs_APEX")
plt.show()

# sets significant in >= 2 cohorts of each data set (GO:CC), activities side by side
db = "GO:CC" if "GO:CC" in DBS else DBS[0]
so = act_o.loc[(act_o["fit"] == db) & (act_o["padj"] < 0.05)].groupby("set_id").size()
sa = act_a.loc[(act_a["fit"] == db) & (act_a["padj"] < 0.05)].groupby("set_id").size()
sets = so.index[so >= 2].intersection(sa.index[sa >= 2])
if len(sets):
    mo = act_o.loc[(act_o["fit"] == db) & act_o["set_id"].isin(sets)].pivot(index="set_id", columns="cohort", values="activity")[[c for c in ONT_COH]]
    ma = act_a.loc[(act_a["fit"] == db) & act_a["set_id"].isin(sets)].pivot(index="set_id", columns="cohort", values="activity")[[c for c in APEX_COH]]
    names = act_o.drop_duplicates("set_id").set_index("set_id")["name"].fillna("").str.slice(0, 60)
    both = pd.concat([mo.add_prefix("ONT "), ma], axis=1)
    both = both.loc[both.abs().max(axis=1).sort_values(ascending=False).index[:40]]
    save_tsv(both.rename(index=names).rename_axis("set").reset_index(), f"gene_set_activities.sig_in_both.{db.replace(':', '_')}.tsv")
    z = both.div(both.abs().max(axis=0), axis=1)          # scaled per cohort: the two data sets differ in the range of activities
    sns.set(font_scale=0.75, style="white")
    g = sns.clustermap(z.rename(index=names), cmap="RdBu_r", center=0, vmin=-1, vmax=1, col_cluster=False, figsize=(14, 0.2 * len(z) + 3),
                       yticklabels=True, cbar_kws={"label": "activity / max |activity| of the cohort"}, dendrogram_ratio=(0.08, 0.05))
    g.ax_heatmap.axvline(len(ONT_COH), color="black", lw=1.5)
    g.ax_heatmap.set(xlabel="", ylabel="")
    for t in g.ax_heatmap.get_xticklabels():
        t.set_color(FRACTION_COLORS.get(frac_of(t.get_text()), "black"))
    g.savefig(os.path.join(FIGS, f"gene_set_activities.sig_in_both.{db.replace(':', '_')}.png"), bbox_inches="tight", dpi=200)
    g.savefig(os.path.join(FIGS, f"gene_set_activities.sig_in_both.{db.replace(':', '_')}.pdf"), bbox_inches="tight")
    figure_caption(os.path.join(FIGS, f"gene_set_activities.sig_in_both.{db.replace(':', '_')}"))
    plt.show()
else:
    print(f"{db}: no set significant in >= 2 cohorts of both data sets")

## 3' UTR length (APA)

$D_g$ (log2 usage-weighted mean 3' UTR length, fraction − reference) per gene: ONT from the cleavage sites (read 3' ends) in the PAQR3 terminal exon, APEX-seq from the PAQR tandem PAS of the terminal exon. Spearman correlation of $D_g$ over the genes tested in both, and the median $D_g$ per cohort.

In [ ]:
def load_apa(folder):
    d = pd.read_csv(os.path.join(folder, "apa", "utr_length.fraction_vs_Total.tsv.gz"), sep="\t")
    d["gene"] = d["gene_id"].str.replace(r"\.\d+$", "", regex=True)
    return d

apa_o, apa_a = load_apa(T_ONT), load_apa(T_APEX)
Do = apa_o.pivot(index="gene", columns="cohort", values="log2FC_length").reindex(columns=ONT_COH)
Da = apa_a.pivot(index="gene", columns="cohort", values="log2FC_length").reindex(columns=APEX_COH)
g_apa = Do.dropna(how="all").index.intersection(Da.dropna(how="all").index)
m_apa = spearman_matrix(Do.loc[g_apa], Da.loc[g_apa])
save_tsv(m_apa.rename_axis("ONT cohort").reset_index(), "spearman_D_utr_length.ONT_vs_APEX.tsv")
med = pd.concat([Do.median().rename("median D_g").to_frame().assign(data="ONT"), Da.median().rename("median D_g").to_frame().assign(data="APEX-seq")])
save_tsv(med.rename_axis("cohort").reset_index(), "median_D_utr_length.per_cohort.tsv")
print(f"{len(g_apa):,} genes with D_g in both data sets")

sns.set(font_scale=0.8, style="white")
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1.4, 1]})
lim = np.nanmax(np.abs(m_apa.values))
sns.heatmap(m_apa, cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, annot=True, fmt=".2f", annot_kws={"fontsize": 6}, ax=axes[0],
            cbar_kws={"label": "Spearman of $D_g$ (3' UTR length)"})
axes[0].set(xlabel="APEX-seq", ylabel="ONT (FACS)", title=f"{len(g_apa):,} genes")
color_ticklabels(axes[0].get_xticklabels()); color_ticklabels(axes[0].get_yticklabels())
ax = axes[1]
y = np.arange(len(med))
ax.barh(y, med["median D_g"], color=[FRACTION_COLORS.get(frac_of(c), "grey") for c in med.index], edgecolor="black", lw=0.3)
ax.set_yticks(y, [f"{'ONT ' if d == 'ONT' else ''}{c}" for c, d in zip(med.index, med["data"])], fontsize=7)
ax.set_ylim(len(med) - 0.5, -0.5)
ax.axvline(0, color="black", lw=0.8)
ax.set(xlabel="median $D_g$ (3' UTR length, fraction - reference)")
sns.despine(ax=ax); fig.tight_layout()
savefig(fig, "apa.ONT_vs_APEX")
plt.show()

## Interpretation

- **Genes.** 3,747 genes have $\alpha$ in both data sets and are expressed in both references (3,539 protein-coding). The cell lines (HCT116, HEK293T), the methods (FACS sorts with ONT, proximity labelling with Illumina) and the references differ, so only correlations of moderate size can be expected.
- **ER-associated mRNAs: the expected pair agrees moderately.**
  - ONT ER correlates with APEX ERM (Spearman 0.29 on protein-coding genes; 0.33 with the nuclear axis removed) and with OMM (0.27), less with KDEL (0.12), and negatively with the cytosolic fractionation (−0.32). Its under-represented genes overlap those of ERM (odds ratio 1.6, p = 0.07).
  - ONT TIS does not correlate with ERM (−0.05), although its under-represented genes overlap (odds ratio 1.7, p = 0.0003).
- **The strongest agreement is between ONT Golgi, PEX, Endo and TJ and APEX ERM / OMM** (Spearman 0.35–0.44), and against the cytosolic fractionation (−0.45 to −0.50). Over- and under-represented genes overlap with odds ratios of 4.5 and 2.0–2.2 (Golgi, Endo vs ERM; p < 0.0001).
  - These are the ER-translated mRNAs: the gene-set heatmap shows "ER lumen" and "ER protein-containing complex" over-represented in Org2, Golgi, PEX, Endo and TJ and in ERM, OMM and Nuc.
  - The FACS-sorted membrane fractions therefore carry rough-ER material, as expected for vesicles and organelle fragments that co-sort with ER membranes.
  - This agrees with the nuclear-envelope ER of the nuclear fraction in APEX-seq.
- **The nuclear axis is weak in this comparison.** ONT Golgi/PEX/Endo/TJ correlate only weakly with APEX NLS, NIK and LMA (0.08–0.22), and not with the nuclear fractionation (−0.08 to 0.00). The cytosolic fractionation, in contrast, is anti-correlated. Part of the reason is that fracSanity flagged the most nuclear RNAs as outliers in both data sets (`APEX_seq.fracSanity.ipynb`), and the comparison is restricted to genes expressed in both references, mostly protein-coding.
- **Free (ONT) vs NES (APEX) agree only weakly** (0.07–0.17). In both, the "cytosolic" sample is close to the reference, so its $r_g$ carry little signal, and none of the NES genes are called.
- **APA.** The 3' UTR length changes ($D_g$, 1,471 genes) of ONT Golgi, PEX, Endo and TJ correlate weakly with those of most APEX cohorts (0.11–0.18), and ONT ER negatively (−0.10 to −0.17). The APEX APA changes follow library complexity (`APEX_seq.fracSanity.ipynb`). The weak positive correlation may reflect genes whose isoforms differ between whole cells and any isolated compartment, but it does not confirm the ONT APA result.

## Summary

- **Both data sets see the ER-translated mRNAs:** FACS-sorted Golgi, PEX, Endo and TJ (and ER, more weakly) share their over- and under-represented mRNAs with APEX ERM and OMM (Spearman up to 0.44; odds ratios 2–4.5). ER-lumen and ER-complex gene sets drive this, and the cytosolic fractionation is depleted of the same mRNAs.
- **The nuclear-RNA axis and the cytosolic match are weak** in the cross-comparison. This is partly because fracSanity discounts the most strongly localised RNAs as outliers.
- **APA does not replicate convincingly:** weak correlation (≤ 0.18), and the APEX APA changes are confounded with library complexity.